In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
path = "/global/homes/b/binxia/scratch/plasma/runs/beta0.01_nu1_Bz0.15_dt2_tau200/CSV_Data/"
# path = "/global/homes/b/binxia/scratch/plasma/runs/beta0.1_nu0_Bz0_dt2_tau40/CSV_Data/"

In [3]:
# for i in range(0, 1):#49):
#     Bx = np.loadtxt(path + f"Bx_{i}.csv", delimiter=',')
#     By = np.loadtxt(path + f"By_{i}.csv", delimiter=',')
#     Bz = np.loadtxt(path + f"Bz_{i}.csv", delimiter=',')
#     Density = np.loadtxt(path + f"Density_{i}.csv", delimiter=',')

#     # 假设 shape = (Nx, Nz)
#     Nx, Nz = Density.shape

#     # 如果你暂时只想用数组 index 当坐标：
#     x = np.arange(Nx)
#     z = np.arange(Nz)

#     # 如果想和之前图上类似的物理坐标范围，可以改成：
#     # x = np.linspace(-50, 50, Nx)
#     # z = np.linspace(-21, 21, Nz)

#     Z, X = np.meshgrid(z, x)

#     # quiver 下采样，否则太密
#     step = 20

#     # 平面内磁场强度
#     B_inplane = np.sqrt(Bx**2 + Bz**2)

#     plt.figure(figsize=(6, 8), dpi=200)

#     # 背景：Density
#     plt.imshow(
#         Density,
#         cmap='plasma',
#         origin='lower',
#         extent=[z.min(), z.max(), x.min(), x.max()],
#         aspect='auto',
#         vmin=0,
#         vmax=4,
#     )
#     plt.colorbar(label='Density')

#     # 磁力线：注意 streamplot 里传的是 (horizontal, vertical) = (Bz, Bx)
#     plt.streamplot(
#         z, x,            # x-coordinates (horizontal), y-coordinates (vertical)
#         Bz, Bx,          # U, V
#         color='white',   # 也可以改成 B_inplane
#         density=1.5,
#         linewidth=0.8,
#         arrowsize=0.8,
#     )

#     # 再叠加稀疏 quiver，帮助看局部方向
#     plt.quiver(
#         Z[::step, ::step],
#         X[::step, ::step],
#         Bz[::step, ::step],   # horizontal component
#         Bx[::step, ::step],   # vertical component
#         color='cyan',
#         scale=15,
#         width=0.002,
#     )


#     # 关键：强制坐标范围回到 density image 的范围
#     plt.xlim(z.min(), z.max())
#     plt.ylim(x.min(), x.max())

#     plt.title(f'Density + in-plane magnetic field, timestep {i}')
#     plt.xlabel('z')
#     plt.ylabel('x')

#     plt.tight_layout()
#     plt.savefig(f'Density_{i:03d}.png')
#     # plt.show()
#     plt.close()

In [4]:
# print(f"{Bx.shape=}, {By.shape=}, {Bz.shape=}, {Density.shape=}")

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FormatStrFormatter

for i in range(0, 149):
    Bx = np.loadtxt(path + f"Bx_{i}.csv", delimiter=',')
    By = np.loadtxt(path + f"By_{i}.csv", delimiter=',')
    Bz = np.loadtxt(path + f"Bz_{i}.csv", delimiter=',')
    Density = np.loadtxt(path + f"Density_{i}.csv", delimiter=',')

    Nx, Nz = Density.shape

    x = np.arange(Nx)
    z = np.arange(Nz)

    Z, X = np.meshgrid(z, x)

    step = 20

    # =========================
    # compute Ay
    # =========================
    Ay = np.zeros_like(Density, dtype=np.float64)

    for ix in range(1, Nx):
        dx = x[ix] - x[ix - 1]
        Ay[ix, 0] = Ay[ix - 1, 0] + 0.5 * (Bz[ix - 1, 0] + Bz[ix, 0]) * dx

    for iz in range(1, Nz):
        dz = z[iz] - z[iz - 1]
        Ay[:, iz] = Ay[:, iz - 1] - 0.5 * (Bx[:, iz - 1] + Bx[:, iz]) * dz

    Ay = Ay - Ay.mean()

    # =========================
    # compute Jy
    # =========================
    dBx_dz = np.gradient(Bx, z, axis=1)
    dBz_dx = np.gradient(Bz, x, axis=0)
    Jy = dBx_dz - dBz_dx

    # jmax = np.percentile(np.abs(Jy), 99)
    levels = 15

    # =========================
    # figure
    # =========================
    fig, axes = plt.subplots(1, 2, figsize=(10, 6), dpi=200)
    # 手动控制子图间距
    fig.subplots_adjust(wspace=0.3)

    # ===== 左图 =====
    ax0 = axes[0]
    jy_vmin = -0.1
    jy_vmax =  0.1

    im0 = ax0.imshow(
        Jy,
        cmap='seismic',
        origin='lower',
        extent=[z.min(), z.max(), x.min(), x.max()],
        aspect='auto',
        vmin=jy_vmin,
        vmax=jy_vmax,
    )
    ax0.contour(
        Z, X, Ay,
        levels=levels,
        colors='black',
        linewidths=0.6,
    )
    ax0.set_xlim(z.min(), z.max())
    ax0.set_ylim(x.min(), x.max())
    ax0.set_title(rf'$J_y$ + $A_y$ contours, timestep {i}')
    ax0.set_xlabel('z')
    ax0.set_ylabel('x')

    # 给左图单独挂 colorbar，长度与子图一致，紧贴子图
    divider0 = make_axes_locatable(ax0)
    cax0 = divider0.append_axes("right", size="4.5%", pad=0.01)
    cbar0 = fig.colorbar(im0, cax=cax0)
    cbar0.set_label(r'$J_y$')
    cbar0.ax.yaxis.set_major_formatter(FormatStrFormatter('%.0e'))
    cbar0.ax.tick_params(labelrotation=90)

    ax0.quiver(
        Z[::step, ::step],
        X[::step, ::step],
        Bz[::step, ::step],
        Bx[::step, ::step],
        color='black',
        scale=15,
        width=0.002,
    )

    # ===== 右图 =====
    ax1 = axes[1]
    im1 = ax1.imshow(
        Density,
        cmap='plasma',
        origin='lower',
        extent=[z.min(), z.max(), x.min(), x.max()],
        aspect='auto',
        vmin=0,
        vmax=4,
    )
    ax1.contour(
        Z, X, Ay,
        levels=levels,
        colors='white',
        linewidths=0.8,
    )
    ax1.set_xlim(z.min(), z.max())
    ax1.set_ylim(x.min(), x.max())
    ax1.set_title(rf'Density + $A_y$ contours + in-plane $B$ field')
    ax1.set_xlabel('z')
    ax1.set_ylabel('x')

    # 给右图单独挂 colorbar
    divider1 = make_axes_locatable(ax1)
    cax1 = divider1.append_axes("right", size="4.5%", pad=0.01)
    cbar1 = fig.colorbar(im1, cax=cax1)
    cbar1.set_label('Density')
    cbar1.ax.tick_params(labelrotation=90)

    plt.savefig(f'Density_{i:03d}.png', bbox_inches='tight')
    plt.close()